# Risky-Debt Model — SMM Estimation (Calibrated $c_{\text{def}}$)

Same setup as `06_risky_debt_smm_workflow.ipynb`, but $c_{\text{def}}$ is fixed at its true value (0.45)
and removed from the estimated parameter vector. This isolates whether the weak identification of
$c_{\text{def}}$ was degrading estimation of the remaining 6 parameters.

In [1]:
import contextlib, io, logging, os, sys, time, warnings
from dataclasses import asdict
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
warnings.filterwarnings("ignore", message=r".*does not produce the same series.*")
logging.getLogger("tensorflow").setLevel(logging.ERROR)

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
from scipy.stats import norm
from IPython.display import display

from src.v2.estimation import SMMMonteCarloConfig, SMMRunConfig, solve_smm
from src.v2.evaluation import prepare_evaluation_run, save_manifest_sections, save_summary_rows
from src.v2.environments.risky_debt import EconomicParams, RiskyDebtEnv, ShockParams
from src.v2.solvers import RiskyDebtSolverConfig, solve_risky_debt
from src.v2.utils.seeding import fold_in_seed

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = "{:.4f}".format

def _run_quietly(fn, *args, **kwargs):
    buf = io.StringIO()
    t0 = time.perf_counter()
    with contextlib.redirect_stdout(buf):
        result = fn(*args, **kwargs)
    return result, time.perf_counter() - t0

# Short labels for display
_ML = {"avg_equity_issuance_assets": "Avg Iss/k",
       "book_leverage": "Book Lev", "cov_leverage_investment": "Cov(Lev,I)",
       "mean_investment_assets": "Mean I/k", "serial_corr_investment": "AutoCorr(I/k)", "var_investment_assets": "Var I/k",
       "income_ar1_beta": "AR1 beta", "income_ar1_resid_std": "AR1 sigma",
       "default_frequency": "Def Freq",
       "frequency_equity_issuance": "Freq Iss",
       "corr_issuance_investment": "Corr(Iss,I)"}
_PL = {"alpha": "alpha", "psi1": "psi1", "eta0": "eta0", "eta1": "eta1",
       "c_def": "c_def", "rho": "rho", "sigma_epsilon": "sigma"}

/Users/wangzhaoxuan/Desktop/JPM-TSRL/DL_corp_finance/.venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
# ── Profile ───────────────────────────────────────────────────────────
PROFILE = "SMOKE"  # "SMOKE" or "FULL"

_P = {
    "SMOKE": dict(solver=dict(n_k=20, n_b=30, n_z=15, n_z_solve=7),
                  smm=dict(n_firms=300, horizon=25, burn_in=25, n_sim_panels=300,
                           global_method="differential_evolution", local_method="Powell",
                           optimizer_maxiter=5, optimizer_popsize=5, master_seed=(20, 26)),
                  mc=dict(n_replications=2)),
    "FULL": dict(solver=dict(n_k=25, n_b=25, n_z=15, n_z_solve=10),
                 smm=dict(n_firms=5000, horizon=25, burn_in=75, n_sim_panels=300,
                          global_method="differential_evolution", local_method="Powell",
                          optimizer_maxiter=5, optimizer_popsize=15, master_seed=(20, 26)),
                 mc=dict(n_replications=5)),
}

# ── Solver: nested VFI with z-interpolation + adaptive b' bounds ──────
ADAPTIVE     = True
BUFFER_FRAC  = 0.10

# ── Structural parameters ─────────────────────────────────────────────
# c_def is CALIBRATED (fixed at true value), not estimated.
ESTIMATED_PARAMS = ("alpha", "psi1", "eta0", "eta1", "rho", "sigma_epsilon")

TRUE_PARAMS = dict(production_elasticity=0.70, cost_convex=0.05,
                   cost_inject_fixed=0.10, cost_inject_linear=0.05,
                   default_haircut=0.45, rho=0.60, sigma=0.15)

CALIBRATED = dict(interest_rate=0.05, depreciation_rate=0.15, tax=0.30,
                  default_haircut=0.45)  # c_def calibrated here

BETA_BOUNDS = ((0.30,0.90), (0.01,0.25), (0.01,0.30), (0.01,0.20),
               (0.30,0.90), (0.05,0.50))
BETA_INITIAL_GUESS = [0.5, 0.13, 0.15, 0.10, 0.55, 0.25]

# ── Build objects ─────────────────────────────────────────────────────
env = RiskyDebtEnv(
    econ_params=EconomicParams(
        interest_rate=CALIBRATED["interest_rate"],
        depreciation_rate=CALIBRATED["depreciation_rate"],
        tax=CALIBRATED["tax"],
        production_elasticity=TRUE_PARAMS["production_elasticity"],
        cost_convex=TRUE_PARAMS["cost_convex"],
        default_haircut=TRUE_PARAMS["default_haircut"],
        cost_inject_fixed=TRUE_PARAMS["cost_inject_fixed"],
        cost_inject_linear=TRUE_PARAMS["cost_inject_linear"],
    ),
    shock_params=ShockParams(mu=0.0, rho=TRUE_PARAMS["rho"], sigma=TRUE_PARAMS["sigma"]),
    k_min_mult=0.05, k_max_mult=1.5, b_max_mult=4.0, b_min_mult=0.2,
)

p = _P[PROFILE]
SOLVER_CONFIG = RiskyDebtSolverConfig(
    **p["solver"],
    adaptive=ADAPTIVE,
    buffer_frac=BUFFER_FRAC,
)
SMM_RUN_CONFIG = SMMRunConfig(**p["smm"])

RUN = prepare_evaluation_run("smm_risky_debt", save_run=True,
                             run_tag="smm-risky-debt-calibrated-cdef",
                             results_root=str(REPO_ROOT / "outputs" / "notebooks"))
save_manifest_sections(RUN, profile=PROFILE, estimated=list(ESTIMATED_PARAMS),
                       calibrated=CALIBRATED, true_params=TRUE_PARAMS,
                       solver=asdict(SOLVER_CONFIG), smm=asdict(SMM_RUN_CONFIG))

print(f"Profile: {PROFILE}")
print(f"Solver:  k={SOLVER_CONFIG.n_k} x b={SOLVER_CONFIG.n_b} x z={SOLVER_CONFIG.n_z} "
      f"(z_solve={SOLVER_CONFIG.n_z_solve})")
print(f"         adaptive={ADAPTIVE}, buffer={BUFFER_FRAC:.0%}")
print(f"SMM:     firms={SMM_RUN_CONFIG.n_firms}, T={SMM_RUN_CONFIG.horizon}, "
      f"S={SMM_RUN_CONFIG.n_sim_panels}, maxiter={SMM_RUN_CONFIG.optimizer_maxiter}")
print(f"Estimated: {ESTIMATED_PARAMS}")
print(f"Calibrated c_def = {CALIBRATED['default_haircut']}")

Profile: SMOKE
Solver:  k=20 x b=30 x z=15 (z_solve=7)
         adaptive=True, buffer=10%
SMM:     firms=300, T=25, S=300, maxiter=5
Estimated: ('alpha', 'psi1', 'eta0', 'eta1', 'rho', 'sigma_epsilon')
Calibrated c_def = 0.45


In [3]:
beta_true = env.smm_true_beta(estimated_params=ESTIMATED_PARAMS)
# Disable diagnostic moments (Jacobian audit shows weak/redundant identification);
# the H&W 2007 issuance & leverage moments below replace them in the active set.
DIAGNOSTIC_MOMENTS = ("conditional_issuance_size", "autocorr_equity_issuance",
                      "crosscorr_leverage_issuance")

spec = env.make_smm_spec(solver_config=SOLVER_CONFIG, initial_guess=BETA_INITIAL_GUESS,
                         bounds=BETA_BOUNDS, estimated_params=ESTIMATED_PARAMS,
                         disabled_moments=DIAGNOSTIC_MOMENTS)

display(pd.DataFrame({
    "param": [_PL[p] for p in spec.parameter_names],
    "true": beta_true, "init": spec.initial_guess,
    "lo": [b[0] for b in spec.bounds], "hi": [b[1] for b in spec.bounds],
}))
print(f"Moments ({len(spec.moment_names)}): {[_ML[m] for m in spec.moment_names]}")

# Reference solve + target moments
solved, t = _run_quietly(solve_risky_debt, env, config=SOLVER_CONFIG)
print(f"\nRef solve: {solved['stop_reason']}, outer={solved['n_outer']}, wall={t:.1f}s")
if "adaptive_b_bounds" in solved:
    print(f"Adaptive b': [{solved['adaptive_b_bounds'][0]:.1f}, {solved['adaptive_b_bounds'][1]:.1f}]")

target_seed = fold_in_seed(SMM_RUN_CONFIG.master_seed, "smm", "risky_debt", "target")
target, t = _run_quietly(spec.simulate_target_moments, beta_true, SMM_RUN_CONFIG, target_seed)
print(f"Target moments: generated in {t:.1f}s")

,param,true,init,lo,hi
0,alpha,0.7000,0.5000,0.3000,0.9000
1,psi1,0.0500,0.1300,0.0100,0.2500
2,eta0,0.1000,0.1500,0.0100,0.3000
3,eta1,0.0500,0.1000,0.0100,0.2000
4,rho,0.6000,0.5500,0.3000,0.9000
5,sigma,0.1500,0.2500,0.0500,0.5000


Moments (8): ['Avg Iss/k', 'Mean I/k', 'AutoCorr(I/k)', 'Var I/k', 'AR1 beta', 'AR1 sigma', 'Freq Iss', 'Corr(Iss,I)']

Ref solve: converged_outer_value, outer=9, wall=7.3s
Adaptive b': [56.8, 94.9]
Target moments: generated in 6.5s


## Diagnostics: Oracle + Jacobian

In [4]:
# Oracle test: Q(beta_true) ≈ 0
oracle_seed = fold_in_seed(SMM_RUN_CONFIG.master_seed, "smm", "oracle")
oracle_result, t = _run_quietly(spec.simulate_panel_moments, beta_true, SMM_RUN_CONFIG, oracle_seed)
oracle_err = oracle_result.panel_moments.mean(axis=0) - target.values
Q = float(oracle_err @ oracle_err)
print(f"Oracle: Q={Q:.2e}, ||e||={np.sqrt(Q):.2e}, wall={t:.1f}s — {'PASS' if Q < 0.01 else 'FAIL'}")

# Jacobian (finite differences)
print(f"\nJacobian: {2*len(beta_true)} solves ...")
fd_steps = 0.01 * (np.array(BETA_BOUNDS)[:, 1] - np.array(BETA_BOUNDS)[:, 0])
K, R = len(beta_true), len(target.values)
jac = np.zeros((R, K))

t0 = time.perf_counter()
for j in range(K):
    bp, bm = beta_true.copy(), beta_true.copy()
    bp[j] += fd_steps[j]; bm[j] -= fd_steps[j]
    mp, _ = _run_quietly(spec.simulate_panel_moments, bp, SMM_RUN_CONFIG, oracle_seed)
    mm, _ = _run_quietly(spec.simulate_panel_moments, bm, SMM_RUN_CONFIG, oracle_seed)
    jac[:, j] = (mp.panel_moments.mean(0) - mm.panel_moments.mean(0)) / (2 * fd_steps[j])

sv = np.linalg.svd(jac, compute_uv=False)
rank = int(np.sum(sv > 1e-8))
jac_wall = time.perf_counter() - t0
print(f"  SV: {sv}")
print(f"  Rank: {rank}/{K} — wall={jac_wall:.1f}s")

# Sensitivity table
ml = [_ML[m] for m in spec.moment_names]
pl = [_PL[p] for p in spec.parameter_names]
norms = np.linalg.norm(jac, axis=0)
status = ["DEAD" if n < 1e-6 else "WEAK" if n < 0.05 else "OK" for n in norms]

sensitivity_df = pd.DataFrame({"param": pl, "||D||": norms, "status": status})
display(sensitivity_df)

jacobian_df = pd.DataFrame(jac, index=ml, columns=pl).round(4)
display(jacobian_df)

dead = [p for p, s in zip(pl, status) if s == "DEAD"]
weak = [p for p, s in zip(pl, status) if s == "WEAK"]
if dead: print(f"Dead: {dead}")
if weak: print(f"Weak: {weak}")

# ── Save diagnostics ─────────────────────────────────────────────────
oracle_info = [{"metric": "Q", "value": Q},
               {"metric": "||e||", "value": float(np.sqrt(Q))},
               {"metric": "oracle_pass", "value": Q < 0.01},
               {"metric": "jacobian_rank", "value": rank},
               {"metric": "jacobian_K", "value": K},
               {"metric": "jacobian_wall_sec", "value": round(jac_wall, 1)}]
for i, s in enumerate(sv):
    oracle_info.append({"metric": f"sv_{i+1}", "value": float(s)})
save_summary_rows(RUN, oracle_info, filename="diagnostics_oracle.csv")
save_summary_rows(RUN, sensitivity_df.to_dict("records"), filename="diagnostics_sensitivity.csv")
save_summary_rows(RUN, jacobian_df.reset_index().rename(columns={"index": "moment"}).to_dict("records"),
                  filename="diagnostics_jacobian.csv")

Oracle: Q=1.36e-03, ||e||=3.69e-02, wall=7.2s — PASS

Jacobian: 12 solves ...
  SV: [15.9033  6.6354  3.4188  1.0155  0.9052  0.0239]
  Rank: 6/6 — wall=74.5s


,param,||D||,status
0,alpha,4.1159,OK
1,psi1,6.8665,OK
2,eta0,6.4755,OK
3,eta1,10.8116,OK
4,rho,8.3631,OK
5,sigma,4.1987,OK


,alpha,psi1,eta0,eta1,rho,sigma
Avg Iss/k,0.1344,-0.5600,-0.4853,-1.0214,-0.1062,-0.1526
Mean I/k,0.0269,-0.3135,-0.2277,-0.2976,-0.0194,0.0427
AutoCorr(I/k),0.5780,0.6111,0.5482,2.9354,-0.7310,-1.5840
Var I/k,0.0544,-0.5833,-0.4014,-0.4970,-0.0347,0.0780
AR1 beta,0.6692,0.1010,-0.1421,-0.1877,-1.2946,-1.0568
AR1 sigma,-0.0063,-0.2526,-0.1851,-0.2090,-0.1730,1.2650
Freq Iss,0.1994,-6.0592,-4.3815,-5.0912,0.0413,-3.3335
"Corr(Iss,I)",-4.0121,-3.0391,-4.6830,-8.9942,-8.2272,-1.1222


PosixPath('/Users/wangzhaoxuan/Desktop/JPM-TSRL/DL_corp_finance/outputs/notebooks/smm_risky_debt/smm-risky-debt-calibrated-cdef/diagnostics_jacobian.csv')

## Two-Step SMM Estimation

In [5]:
smm_result, wall = _run_quietly(solve_smm, spec, target, SMM_RUN_CONFIG)

# Table 1: Moment fit
moment_fit = pd.DataFrame({
    "moment": [_ML[m] for m in spec.moment_names],
    "target": smm_result.target_moments,
    "fitted": smm_result.stage2.average_moments,
})
print("Moment Fit"); display(moment_fit)

# Table 2: Parameter estimates
beta_hat = smm_result.beta_hat
se = smm_result.standard_errors
t_stat = (beta_hat - beta_true) / np.where(se > 0, se, np.nan)

param_est = pd.DataFrame({
    "param": [_PL[p] for p in spec.parameter_names],
    "true": beta_true, "est": beta_hat, "se": se, "t": t_stat,
    "p": 2 * (1 - norm.cdf(np.abs(t_stat))),
})
print("\nParameter Estimates"); display(param_est)

print(f"\nJ={smm_result.j_statistic:.4f}, p={smm_result.j_p_value:.4f}, df={smm_result.j_df}")
print(f"Stage 1: {SMM_RUN_CONFIG.global_method} nfev={smm_result.stage1.optimizer_nfev}")
print(f"Stage 2: Powell nfev={smm_result.stage2.optimizer_nfev}")
print(f"Wall: {wall:.1f}s")

save_summary_rows(RUN, moment_fit.to_dict("records"), filename="moment_fit.csv")
save_summary_rows(RUN, param_est.to_dict("records"), filename="parameter_estimates.csv")

est_info = [
    {"metric": "j_statistic", "value": smm_result.j_statistic},
    {"metric": "j_p_value", "value": smm_result.j_p_value},
    {"metric": "j_df", "value": smm_result.j_df},
    {"metric": "stage1_method", "value": SMM_RUN_CONFIG.global_method},
    {"metric": "stage1_nfev", "value": smm_result.stage1.optimizer_nfev},
    {"metric": "stage2_method", "value": "Powell"},
    {"metric": "stage2_nfev", "value": smm_result.stage2.optimizer_nfev},
    {"metric": "wall_sec", "value": round(wall, 1)},
]
save_summary_rows(RUN, est_info, filename="estimation_info.csv")

Moment Fit


,moment,target,fitted
0,Avg Iss/k,0.0219,0.0143
1,Mean I/k,0.1633,0.1578
2,AutoCorr(I/k),-0.0055,0.0284
3,Var I/k,0.0275,0.0159
4,AR1 beta,0.3906,0.3963
5,AR1 sigma,0.2170,0.2302
6,Freq Iss,0.2344,0.2630
7,"Corr(Iss,I)",-0.3324,-0.3223



Parameter Estimates


,param,true,est,se,t,p
0,alpha,0.7000,0.7585,0.0031,19.0720,0.0000
1,psi1,0.0500,0.0810,0.0051,6.1155,0.0000
2,eta0,0.1000,0.2316,342970.6439,0.0000,1.0000
3,eta1,0.0500,0.0549,224704.9046,0.0000,1.0000
4,rho,0.6000,0.5125,0.0039,-22.6126,0.0000
5,sigma,0.1500,0.1591,0.0014,6.4554,0.0000



J=44.8614, p=0.0000, df=2
Stage 1: differential_evolution nfev=379
Stage 2: Powell nfev=78
Wall: 2752.1s


PosixPath('/Users/wangzhaoxuan/Desktop/JPM-TSRL/DL_corp_finance/outputs/notebooks/smm_risky_debt/smm-risky-debt-calibrated-cdef/estimation_info.csv')